In [39]:
Purpose:
Clean Orders Data

Operations:
1. Read Bronze Orders
2. Convert String Timestamp
3. Extract Date Components
4. Handle Null Values
5. Save Silver Layer

SyntaxError: invalid syntax (2571211694.py, line 1)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    to_timestamp,
    year,
    month,
    dayofmonth,
    quarter,
    date_format
)

In [ ]:
spark =( 
    SparkSession.builder
    .appName("Clean Order")
    .master("local[*]")
    .getOrCreate()
)



In [ ]:
# Read Bronze Layer
orders_df = spark.read.parquet(
    "output/bronze/orders"
)

In [ ]:
# Initial Inspection
print("\nOriginal Row Count:")
print(orders_df.count())

print("\nSchema:")
orders_df.printSchema()

print("\nSample Data:")
orders_df.show(5, truncate=False)



Original Row Count:
99441

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


Sample Data:
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+-----

In [ ]:
# Remove Duplicate Records

orders_df = orders_df.dropDuplicates()

In [ ]:
 #Remove Invalid Order IDs
 orders_df = orders_df.filter(col("order_id").isNotNull())

In [ ]:
# Convert Purchase Timestamp

orders_df =orders_df.withColumn("purchase_timestamp",to_timestamp(col("order_purchase_timestamp")))

In [ ]:
# Extract Year
orders_df = orders_df.withColumn("purchase_year",year("purchase_timestamp"))

In [ ]:
# Extract Month
orders_df =orders_df.withColumn("purchase_month",month("purchase_timestamp"))

In [ ]:
# Extract Day
orders_df = orders_df.withColumn("purchase_day",dayofmonth("purchase_timestamp"))

In [ ]:
# Extract Quarter
orders_df=orders_df.withColumn("purchase_quarter",quarter("purchase_timestamp"))

In [ ]:
# Extract Weekday Name
orders_df = orders_df.withColumn(
    "purchase_weekday",
    date_format(
        "purchase_timestamp",
        "EEEE"
    )
)

In [ ]:
# Check Null Purchase Dates
null_dates = orders_df.filter(
    col("purchase_timestamp").isNull()
).count()

print(
    f"\nNull Purchase Dates: {null_dates}"
)


Null Purchase Dates: 0


In [ ]:
# Show Original Data Before Transformation
# ====================================

print("\nBEFORE TRANSFORMATION")

orders_df.select(
    "order_id",
    "order_purchase_timestamp"
).show(
    15,
    truncate=False
)


BEFORE TRANSFORMATION
+--------------------------------+------------------------+
|order_id                        |order_purchase_timestamp|
+--------------------------------+------------------------+
|05ce664903d14969bd94d9633ffd8f14|2018-01-09 00:16:48     |
|4b2c34f7fd82d07c39e5072079ffd2a9|2018-04-25 10:32:13     |
|d6c877d8995925578fdeb1d186915549|2017-05-07 12:50:03     |
|f80549a97eb203e1566e026ab66f045b|2017-09-12 10:31:42     |
|c2f3a413b93e57befdbc7877ae75758e|2017-06-27 11:29:08     |
|3ab0d24105c6aa5de67bd849dc3691f9|2018-01-17 16:55:55     |
|0fe75a9839e7296629a6d6924fdf3365|2017-09-03 20:34:23     |
|ab4e9c7509c66adaea2fe37d7e1537c4|2018-07-24 23:00:46     |
|f9f6e986125412f95569a18406c749f0|2018-06-15 00:25:25     |
|7144fcbeada7426f9bb134795620511c|2017-03-27 17:26:43     |
|9b045cde06f6adb6f6187e2119c89869|2017-06-26 10:55:20     |
|c43ac850cb9c3f22945c9466b0c20888|2018-04-12 16:46:36     |
|734a3c56683a8a7dcb4cf7f92319d101|2017-11-25 12:20:11     |
|2d3ae83143613463

In [ ]:
# Display Transformed Data
# ====================================

print("\nTransformed Data:")

orders_df.select(
    "order_id",
    "purchase_timestamp",
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_quarter",
    "purchase_weekday"
).show(15, truncate=False)


Transformed Data:
+--------------------------------+-------------------+-------------+--------------+------------+----------------+----------------+
|order_id                        |purchase_timestamp |purchase_year|purchase_month|purchase_day|purchase_quarter|purchase_weekday|
+--------------------------------+-------------------+-------------+--------------+------------+----------------+----------------+
|fd1b1552b93d46b554afde387322f286|2018-06-09 09:59:35|2018         |6             |9           |2               |Saturday        |
|e93c226c60a57236abb9d0962b440be9|2017-11-27 17:00:34|2017         |11            |27          |4               |Monday          |
|6646811f070a6e46f85c938eb20b1842|2017-11-26 11:01:04|2017         |11            |26          |4               |Sunday          |
|72a96400285f2bd8dd6d47e55e203b92|2018-03-24 16:36:05|2018         |3             |24          |1               |Saturday        |
|2e10c534d5d49b8fcea5e034821a96e9|2018-02-14 18:41:02|2018      